# Fase 1: Análisis Exploratorio de Datos (EDA) e Inferencia Estadística
**Proyecto:** Reducción de Variabilidad en Neumáticos Mineros - Flota CAT 797F
**Autor:** Ruben Ronald Bautista Huillca

En este notebook realizaremos la ingesta del archivo Excel original, procesaremos los parámetros operativos y ejecutaremos una prueba de hipótesis para cuantificar el impacto geomecánico entre el Tajo Norte y el Tajo Sur.

In [19]:
# Importación de librerías esenciales
import pandas as pd
import numpy as np
from scipy import stats
import warnings

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


### 1. Ingesta de Datos desde el archivo Excel
Leeremos directamente las hojas `00 Estudio` y `01 Parametros` desde nuestro directorio `data/raw/`.

In [20]:
# Ruta del archivo (ajusta si es necesario según tu estructura de carpetas)
ruta_excel = '../data/raw/00 Data (12).xlsx'

# Carga de hojas
df_estudio = pd.read_excel(ruta_excel, sheet_name='00 Estudio')
df_parametros = pd.read_excel(ruta_excel, sheet_name='01 Parametros')
print("Datos cargados exitosamente.")
print(f"Dimensiones de la muestra de estudio: {df_estudio.shape}")

# Damos un vistazo rápido a los datos de estudio
display(df_estudio.head(3))

Datos cargados exitosamente.
Dimensiones de la muestra de estudio: (10, 8)


,ID_Camion_Test,Tajo_Asignado,Periodo_Prueba,Horas_Trabajadas,Profundidad_Inicial_mm,Profundidad_Final_mm,Vida_Util_Proyectada_horas,Condiciones
0,TEST_001,Tajo_1 (Tajo Norte),Jul-Oct 2024 (4 meses),1424.8,65,55.8,6201,"Roca 70 MPa, +10% pendiente, sube cargado"
1,TEST_002,Tajo_1 (Tajo Norte),Jul-Oct 2024 (4 meses),1393.1,65,56.0,6201,"Roca 70 MPa, +10% pendiente, sube cargado"
2,TEST_003,Tajo_1 (Tajo Norte),Jul-Oct 2024 (4 meses),1432.4,65,55.8,6201,"Roca 70 MPa, +10% pendiente, sube cargado"


### 2. Extracción de Parámetros Clave
Antes de calcular los costos, necesitamos extraer dinámicamente el precio de los neumáticos desde la hoja de parámetros. Esto hace que el código sea escalable si gerencia actualiza el Excel en el futuro.

In [21]:
# Extraer el costo unitario de la llanta (limpiando el signo de dólar y comas si los hay)
# Buscamos la fila donde el 'Parámetro' es 'Precio Unitario (USD)'
fila_precio = df_parametros[df_parametros['Parámetro'] == 'Precio Unitario (USD)']['Valor'].values[0]

# Limpiamos el string "$52,000" a float 52000.0
precio_unitario = float(str(fila_precio).replace('$', '').replace(',', ''))

# Costo total por camión (6 llantas)
costo_total_llantas = precio_unitario * 6

print(f"Precio Unitario por Llanta extraído: ${precio_unitario:,.2f}")
print(f"Costo Total de Llantas por Camión: ${costo_total_llantas:,.2f}")

Precio Unitario por Llanta extraído: $52,000.00
Costo Total de Llantas por Camión: $312,000.00


### 3. Feature Engineering (Ingeniería de Características)
Calcularemos las variables exigidas por las instrucciones:
1. **Tasa de desgaste ($mm/h$)** = (Profundidad Inicial - Profundidad Final) / Horas Trabajadas
2. **Costo por Hora (CPH)** = Costo Total del Juego de Llantas / Vida Útil Proyectada

In [22]:
# 1. Cálculo de Desgaste total en mm
df_estudio['Desgaste_mm'] = df_estudio['Profundidad_Inicial_mm'] - df_estudio['Profundidad_Final_mm']

# 2. Cálculo de la Tasa de Desgaste (mm/h)
df_estudio['Tasa_Desgaste_mm_h'] = df_estudio['Desgaste_mm'] / df_estudio['Horas_Trabajadas']

# 3. Cálculo de CPH Proyectado
df_estudio['CPH_Proyectado'] = costo_total_llantas / df_estudio['Vida_Util_Proyectada_horas']

print("Nuevas variables calculadas:")
display(df_estudio[['ID_Camion_Test', 'Tajo_Asignado', 'Tasa_Desgaste_mm_h', 'CPH_Proyectado']].head())

Nuevas variables calculadas:


,ID_Camion_Test,Tajo_Asignado,Tasa_Desgaste_mm_h,CPH_Proyectado
0,TEST_001,Tajo_1 (Tajo Norte),0.006457,50.314465
1,TEST_002,Tajo_1 (Tajo Norte),0.006460,50.314465
2,TEST_003,Tajo_1 (Tajo Norte),0.006423,50.314465
3,TEST_004,Tajo_1 (Tajo Norte),0.006435,50.314465
4,TEST_005,Tajo_1 (Tajo Norte),0.006483,50.314465
